# MHC-Coach Test

This notebook tests only the code path used for `SriyaM/MHC-Coach` in `nudge-generation`.

- Provider path: Hugging Face (non-MLX)
- Runtime path: `transformers` + `torch`
- Goal: confirm model load + generation succeeds in Colab


In [ ]:
# Install runtime deps for Colab
!pip -q install -U transformers accelerate sentencepiece safetensors

In [ ]:
import json
from typing import Any, Optional

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "SriyaM/MHC-Coach"

In [ ]:
def resolve_torch_device() -> str:
    # Mirrors python_service/huggingface_service.py
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def build_transformers_prompt(tokenizer: Any, prompt: str, device: str):
    # Mirrors python_service/huggingface_service.py
    if hasattr(tokenizer, "apply_chat_template"):
        messages = [{"role": "user", "content": prompt}]
        tokenized = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt",
        )
    else:
        tokenized = tokenizer(prompt, return_tensors="pt")

    if isinstance(tokenized, dict):
        moved = {key: value.to(device) for key, value in tokenized.items()}
        prompt_tokens = moved["input_ids"].shape[-1]
        return moved, prompt_tokens

    moved = tokenized.to(device)
    prompt_tokens = moved.shape[-1]
    return {"input_ids": moved}, prompt_tokens


def generate_transformers_text(
    model,
    tokenizer,
    prompt: str,
    max_tokens: int = 512,
    temperature: Optional[float] = 0.7,
):
    # Mirrors non-MLX branch in python_service/huggingface_service.py
    device = resolve_torch_device()
    inputs, prompt_tokens = build_transformers_prompt(tokenizer, prompt, device)

    do_sample = temperature is not None and temperature > 0
    eos_token_id = tokenizer.eos_token_id
    pad_token_id = tokenizer.pad_token_id
    resolved_pad_token_id = (
        eos_token_id
        if eos_token_id is not None
        else pad_token_id
        if pad_token_id is not None
        else 0
    )

    generation_kwargs = {
        "max_new_tokens": max_tokens,
        "do_sample": do_sample,
        "pad_token_id": resolved_pad_token_id,
    }
    if do_sample:
        generation_kwargs["temperature"] = float(temperature)

    with torch.no_grad():
        output_ids = model.generate(**inputs, **generation_kwargs)

    completion_ids = output_ids[0][prompt_tokens:]
    decoded = tokenizer.decode(completion_ids, skip_special_tokens=True)
    completion = "".join(decoded).strip() if isinstance(decoded, list) else str(decoded).strip()
    return completion


In [ ]:
device = resolve_torch_device()
dtype = torch.float16 if device in {"cuda", "mps"} else torch.float32
print(f"Using device: {device}, dtype: {dtype}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=dtype)
model.to(device)
model.eval()
print("Model and tokenizer loaded")

In [ ]:
# Build a single prompt using shared prompt constants.
# This keeps notebook prompt wording aligned with TypeScript/Python scripts.
from pathlib import Path

prompt_spec_path = Path("config/prompts/prompt_constants.v1.json")
if not prompt_spec_path.exists():
    raise FileNotFoundError(
        f"Could not find {prompt_spec_path}. Run this notebook from the nudge-generation directory."
    )

prompt_spec = json.loads(prompt_spec_path.read_text(encoding="utf-8"))

context = {
    "language": "en",
    "genderIdentity": "female",
    "ageGroup": "35-50",
    "disease": "Diabetes",
    "stageOfChange": "Preparation",
    "educationLevel": "college",
    "preferredNotificationTime": "7:00 AM",
}


def snippet(category: str, key: str | None) -> str:
    defaults = prompt_spec["defaults"]
    if not key:
        return defaults[category]
    return prompt_spec["contextSnippets"][category].get(key, defaults[category])


notification_time_context = prompt_spec["templates"]["notificationTimeContext"].replace(
    "{{preferredNotificationTime}}", context["preferredNotificationTime"]
)

prompt = prompt_spec["templates"]["fullPromptAssembly"]
for placeholder, value in {
    "baseNudgeInstruction": prompt_spec["baseNudgeInstruction"],
    "languageContext": snippet("language", context["language"]),
    "genderContext": snippet("genderIdentity", context["genderIdentity"]),
    "ageContext": snippet("ageGroup", context["ageGroup"]),
    "diseaseContext": snippet("disease", context["disease"]),
    "stageContext": snippet("stageOfChange", context["stageOfChange"]),
    "educationContext": snippet("educationLevel", context["educationLevel"]),
    "notificationTimeContext": notification_time_context,
}.items():
    prompt = prompt.replace(f"{{{{{placeholder}}}}}", value)

response_text = generate_transformers_text(
    model=model,
    tokenizer=tokenizer,
    prompt=prompt,
    max_tokens=512,
    temperature=0.7,
)

print(response_text[:2000])
print("\n--- response length:", len(response_text))

In [ ]:
# Optional: quick JSON validity check (best effort)
try:
    parsed = json.loads(response_text)
    print("Valid JSON:", isinstance(parsed, list), "items:", len(parsed) if isinstance(parsed, list) else "n/a")
except Exception as exc:
    print("Not valid JSON:", exc)